In [1]:
import os
import cv2 as cv
import numpy as np
import pandas as pd
from pathlib import Path
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import entropy

Kode di atas digunakan untuk mendukung proses ekstraksi fitur tekstur citra menggunakan metode Gray Level Co-occurrence Matrix (GLCM). Adapun library yang digunakan beserta kegunaannya dalam file ini adalah sebagai berikut:

a. os
Library os digunakan untuk membantu pengelolaan file dan direktori pada sistem. Namun, pada file ini penggunaannya tidak terlihat secara langsung karena pengelolaan lokasi file lebih banyak menggunakan Path.

b. cv2 (OpenCV)
Library cv2 digunakan untuk membaca gambar hasil preprocessing dalam format grayscale sebelum dilakukan proses ekstraksi fitur tekstur. Pada file ini, OpenCV berfungsi sebagai pembaca citra yang nantinya akan diproses menggunakan metode GLCM.

c. numpy (np)
Library numpy digunakan untuk membantu proses perhitungan numerik pada citra. Dalam file ini, NumPy digunakan untuk menentukan nilai sudut pada GLCM seperti 0°, 45°, 90°, dan 135°, melakukan konversi tipe data gambar menjadi uint8, serta membantu validasi nilai hasil ekstraksi seperti pengecekan nilai NaN atau Infinity.

d. pandas (pd)
Library pandas digunakan untuk menyusun hasil ekstraksi fitur ke dalam bentuk tabel (*dataframe*). Pada file ini, seluruh nilai fitur tekstur dari setiap gambar dikumpulkan menjadi dataset sebelum disimpan ke file CSV.

e. Path (pathlib)
Library Path digunakan untuk menentukan lokasi folder input dan output. Dalam file ini, Path dipakai untuk membaca folder hasil preprocessing serta menentukan lokasi penyimpanan hasil ekstraksi fitur sehingga proses dapat berjalan otomatis untuk seluruh dataset gambar.

f. graycomatrix
Fungsi graycomatrix digunakan untuk membentuk matriks Gray Level Co-occurrence Matrix (GLCM) dari citra grayscale. Dalam file ini, fungsi tersebut menjadi bagian utama proses ekstraksi fitur karena digunakan untuk menganalisis hubungan antar piksel berdasarkan jarak dan sudut tertentu.

g. graycoprops
Fungsi graycoprops digunakan untuk menghitung nilai fitur tekstur berdasarkan matriks GLCM yang telah terbentuk. Pada file ini, fitur yang dihitung meliputi contrast, correlation, homogeneity, dissimilarity, ASM, dan energy sebagai karakteristik tekstur dari setiap citra.

h. entropy
Fungsi entropy digunakan untuk menghitung tingkat kompleksitas atau ketidakteraturan tekstur pada citra berdasarkan matriks yang dihasilkan. Dalam file ini, entropy digunakan sebagai salah satu fitur tambahan untuk merepresentasikan pola tekstur gambar.


In [2]:
from pathlib import Path

# Folder input: hasil preprocessing
PREPROCESSING_DIR = Path("../preprocessing_output")

# Folder output: hasil ekstraksi fitur CSV
OUTPUT_DIR = Path("../hasil_ekstraksi")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Folder hasil preprocessing yang akan diekstraksi fiturnya
PREPO_CONFIG = {
    "prepo1_resize+grayscale": "hasil_ekstraksi_prepo1.csv",
    "prepo2_resize+grayscale+median": "hasil_ekstraksi_prepo2.csv",
    "prepo3_resize+grayscale+median+equ": "hasil_ekstraksi_prepo3.csv",
    "prepo4_resize+grayscale+median+sobel": "hasil_ekstraksi_prepo4.csv",
    "prepo5_resize+grayscale+median+sobel+thresholding": "hasil_ekstraksi_prepo5.csv"
}

VALID_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp"]

print("Folder preprocessing:", PREPROCESSING_DIR.resolve())
print("Folder hasil ekstraksi:", OUTPUT_DIR.resolve())

for prepo_name in PREPO_CONFIG:
    prepo_path = PREPROCESSING_DIR / prepo_name
    print(prepo_name, "->", "ADA" if prepo_path.exists() else "TIDAK ADA")

Folder preprocessing: D:\ASUS\Documents\SEMS 4\Praktikum PCD\Projek\Project-PCD-Kelompok-16\preprocessing_output
Folder hasil ekstraksi: D:\ASUS\Documents\SEMS 4\Praktikum PCD\Projek\Project-PCD-Kelompok-16\hasil_ekstraksi
prepo1_resize+grayscale -> ADA
prepo2_resize+grayscale+median -> ADA
prepo3_resize+grayscale+median+equ -> ADA
prepo4_resize+grayscale+median+sobel -> ADA
prepo5_resize+grayscale+median+sobel+thresholding -> ADA


Berikut penjelasan yang lebih formal dan **pure huruf saja** tanpa tanda seperti bullet atau simbol:

Kode di atas digunakan untuk mengatur lokasi folder input dan output pada proses ekstraksi fitur citra serta melakukan pengecekan terhadap folder preprocessing yang akan digunakan. Pada bagian awal kode dilakukan penentuan folder input yang berisi hasil preprocessing citra menggunakan variabel `PREPROCESSING_DIR`. Folder ini digunakan sebagai sumber data gambar yang nantinya akan diekstraksi fiturnya menggunakan metode Gray Level Co occurrence Matrix atau GLCM.

Selanjutnya program menentukan folder output menggunakan variabel `OUTPUT_DIR` yang berfungsi sebagai tempat penyimpanan hasil ekstraksi fitur dalam bentuk file CSV. Pada bagian ini program juga memastikan bahwa folder hasil ekstraksi tersedia. Jika folder belum ada maka sistem akan membuat folder secara otomatis agar proses penyimpanan hasil tidak mengalami kendala.

Kode juga mendefinisikan konfigurasi preprocessing melalui variabel `PREPO_CONFIG`. Variabel ini berisi daftar folder preprocessing yang akan diproses beserta nama file CSV hasil ekstraksinya. Setiap folder preprocessing merepresentasikan tahapan pengolahan citra yang berbeda mulai dari resize dan grayscale kemudian dilanjutkan dengan median filter equalization sobel hingga thresholding. Dengan adanya konfigurasi ini program dapat memproses beberapa hasil preprocessing secara otomatis tanpa perlu menuliskan ulang kode untuk setiap folder.

Selain itu program menentukan format file gambar yang dapat diproses melalui variabel `VALID_EXTENSIONS`. Pada bagian ini sistem hanya menerima file gambar dengan format JPG JPEG PNG dan BMP sehingga file yang tidak sesuai format tidak akan ikut diproses pada tahap ekstraksi fitur.

Pada bagian akhir kode program menampilkan lokasi folder preprocessing dan folder hasil ekstraksi untuk memastikan direktori yang digunakan sudah benar. Setelah itu sistem melakukan pengecekan terhadap setiap folder preprocessing yang telah didefinisikan sebelumnya untuk mengetahui apakah folder tersedia atau tidak. Hasil pengecekan ini membantu memastikan bahwa seluruh data citra yang diperlukan sudah tersedia sebelum proses ekstraksi fitur dijalankan.


In [3]:
def glcm(image, derajat):
    if derajat == 0:
        angles = [0]
    elif derajat == 45:
        angles = [np.pi / 4]
    elif derajat == 90:
        angles = [np.pi / 2]
    elif derajat == 135:
        angles = [3 * np.pi / 4]
    else:
        raise ValueError("Derajat harus salah satu dari: 0, 45, 90, 135.")

    matriks_glcm = graycomatrix(
        image,
        distances=[1],
        angles=angles,
        levels=256,
        symmetric=True,
        normed=True
    )

    return matriks_glcm


def safe_value(value):
    value = float(value)
    if np.isnan(value) or np.isinf(value):
        return 0.0
    return value


def correlation(matriks):
    return safe_value(graycoprops(matriks, 'correlation')[0, 0])


def dissimilarity(matriks):
    return safe_value(graycoprops(matriks, 'dissimilarity')[0, 0])


def homogenity(matriks):
    return safe_value(graycoprops(matriks, 'homogeneity')[0, 0])


def contrast(matriks):
    return safe_value(graycoprops(matriks, 'contrast')[0, 0])


def ASM(matriks):
    return safe_value(graycoprops(matriks, 'ASM')[0, 0])


def energy(matriks):
    return safe_value(graycoprops(matriks, 'energy')[0, 0])


def entropyGlcm(matriks):
    return safe_value(entropy(matriks.ravel()))

Siap, aku hapus bagian “dengan demikian” dan bikin tetap formal buat laporan:

Kode di atas digunakan untuk membentuk matriks Gray Level Co occurrence Matrix atau GLCM serta menghitung berbagai fitur tekstur yang digunakan pada proses ekstraksi ciri citra. Pada bagian ini program membuat beberapa fungsi khusus untuk menghitung karakteristik tekstur berdasarkan matriks GLCM sehingga proses ekstraksi fitur dapat dilakukan secara terstruktur dan berulang untuk setiap gambar.

Fungsi `glcm` digunakan untuk membentuk matriks GLCM dari citra grayscale berdasarkan sudut tertentu. Pada fungsi ini program menentukan arah hubungan antar piksel menggunakan empat sudut yaitu nol derajat empat puluh lima derajat sembilan puluh derajat dan seratus tiga puluh lima derajat. Setelah sudut ditentukan program membentuk matriks GLCM menggunakan fungsi `graycomatrix` dengan jarak antar piksel sebesar satu piksel. Matriks ini digunakan sebagai dasar dalam perhitungan fitur tekstur citra.

Fungsi `safe_value` digunakan untuk memeriksa hasil perhitungan fitur agar tidak menghasilkan nilai yang tidak valid seperti NaN atau Infinity. Apabila ditemukan nilai tersebut maka sistem akan menggantinya dengan nol sehingga proses ekstraksi fitur dapat berjalan dengan stabil dan data hasil ekstraksi tetap konsisten.

Fungsi `correlation` digunakan untuk menghitung nilai correlation dari matriks GLCM. Nilai correlation menunjukkan tingkat hubungan antar piksel pada citra sehingga dapat menggambarkan pola keterkaitan tekstur yang terbentuk pada gambar.

Fungsi `dissimilarity` digunakan untuk menghitung nilai dissimilarity yang menunjukkan tingkat perbedaan intensitas antar piksel yang berdekatan. Nilai ini membantu menggambarkan variasi tekstur pada citra dimana semakin besar nilainya maka semakin tinggi tingkat perbedaan tekstur gambar.

Fungsi `homogenity` digunakan untuk menghitung tingkat homogenitas tekstur citra. Nilai homogenity menunjukkan keseragaman distribusi piksel pada gambar sehingga semakin tinggi nilainya maka tekstur citra cenderung lebih seragam.

Fungsi `contrast` digunakan untuk menghitung tingkat kontras pada tekstur citra berdasarkan perbedaan intensitas antar piksel. Nilai contrast digunakan untuk menggambarkan tingkat variasi atau ketajaman tekstur pada gambar.

Fungsi `ASM` digunakan untuk menghitung Angular Second Moment yang merepresentasikan tingkat keseragaman pola tekstur citra. Nilai ASM yang tinggi menunjukkan pola tekstur yang lebih teratur dan homogen.

Fungsi `energy` digunakan untuk menghitung nilai energy dari matriks GLCM. Nilai ini menggambarkan kekuatan pola tekstur atau tingkat keteraturan distribusi piksel pada citra.

Fungsi `entropyGlcm` digunakan untuk menghitung nilai entropy dari matriks GLCM. Nilai entropy digunakan untuk mengukur tingkat kompleksitas atau ketidakteraturan tekstur pada citra dimana semakin tinggi nilainya maka pola tekstur gambar cenderung semakin kompleks dan bervariasi.

Kode ini berfungsi untuk menghasilkan berbagai nilai fitur tekstur citra yang nantinya digunakan sebagai karakteristik atau representasi data gambar pada proses analisis maupun klasifikasi.


In [4]:
def extract_glcm_features(image):
    angles = [0, 45, 90, 135]
    features = {}

    for angle in angles:
        matriks = glcm(image, angle)

        features[f"Contrast{angle}"] = contrast(matriks)
        features[f"Homogeneity{angle}"] = homogenity(matriks)
        features[f"Dissimilarity{angle}"] = dissimilarity(matriks)
        features[f"Entropy{angle}"] = entropyGlcm(matriks)
        features[f"ASM{angle}"] = ASM(matriks)
        features[f"Energy{angle}"] = energy(matriks)
        features[f"Correlation{angle}"] = correlation(matriks)

    return features

Kode di atas digunakan untuk melakukan proses **ekstraksi fitur tekstur citra menggunakan metode Gray Level Co occurrence Matrix atau GLCM** berdasarkan beberapa sudut analisis. Pada bagian ini program membuat fungsi `extract_glcm_features` yang bertugas menghitung seluruh fitur tekstur dari sebuah citra secara otomatis. Fungsi ini mempermudah proses ekstraksi karena semua perhitungan fitur dilakukan dalam satu proses tanpa perlu memanggil fungsi satu per satu.

Pada awal fungsi program menentukan empat sudut analisis yaitu nol derajat empat puluh lima derajat sembilan puluh derajat dan seratus tiga puluh lima derajat. Keempat sudut tersebut digunakan untuk menganalisis hubungan antar piksel dari arah yang berbeda sehingga karakteristik tekstur citra dapat diperoleh secara lebih lengkap.

Program kemudian membuat variabel `features` yang digunakan untuk menyimpan seluruh hasil ekstraksi fitur dalam bentuk pasangan nama fitur dan nilai fitur. Setelah itu sistem melakukan perulangan pada setiap sudut yang telah ditentukan. Pada setiap sudut program terlebih dahulu membentuk matriks GLCM menggunakan fungsi `glcm` yang telah dibuat sebelumnya.

Setelah matriks GLCM berhasil dibentuk program menghitung berbagai karakteristik tekstur menggunakan fungsi fungsi yang telah didefinisikan sebelumnya. Fitur yang dihitung meliputi **contrast** untuk mengetahui tingkat perbedaan intensitas antar piksel **homogeneity** untuk mengukur tingkat keseragaman tekstur **dissimilarity** untuk mengetahui tingkat perbedaan tekstur **entropy** untuk mengukur kompleksitas pola tekstur **ASM** untuk mengetahui tingkat keteraturan pola **energy** untuk menggambarkan kekuatan pola tekstur dan **correlation** untuk mengukur hubungan antar piksel pada citra.

Setiap hasil perhitungan fitur disimpan ke dalam variabel `features` dengan nama yang disesuaikan berdasarkan sudut analisis. Sebagai contoh nilai contrast pada sudut nol derajat disimpan dengan nama `Contrast0` sedangkan pada sudut sembilan puluh derajat disimpan dengan nama `Contrast90`. Hal ini dilakukan agar setiap fitur dari masing masing sudut dapat dibedakan dan digunakan pada tahap analisis berikutnya.

Pada bagian akhir fungsi program mengembalikan seluruh hasil ekstraksi fitur yang telah tersimpan dalam variabel `features`. Hasil tersebut nantinya digunakan sebagai representasi karakteristik tekstur citra dan disimpan ke dalam dataset untuk proses analisis atau klasifikasi lebih lanjut.


In [5]:
def extract_features_from_preprocessing_folder(prepo_name, output_csv_name):
    prepo_path = PREPROCESSING_DIR / prepo_name

    if not prepo_path.exists():
        print(f"Folder tidak ditemukan: {prepo_path}")
        return None

    rows = []

    for class_folder in sorted(prepo_path.iterdir()):
        if not class_folder.is_dir():
            continue

        label = class_folder.name

        image_files = [
            file for file in sorted(class_folder.iterdir())
            if file.suffix.lower() in VALID_EXTENSIONS
        ]

        print(f"{prepo_name} | {label}: {len(image_files)} gambar")

        for image_path in image_files:
            image = cv.imread(str(image_path), cv.IMREAD_GRAYSCALE)

            if image is None:
                print(f"Gagal membaca gambar: {image_path}")
                continue

            image = image.astype(np.uint8)

            feature_data = extract_glcm_features(image)

            row = {
                "Filename": image_path.name,
                "Label": label,
                "Preprocessing": prepo_name
            }

            row.update(feature_data)
            rows.append(row)

    df = pd.DataFrame(rows)

    output_csv_path = OUTPUT_DIR / output_csv_name
    df.to_csv(output_csv_path, index=False)

    print(f"CSV berhasil disimpan: {output_csv_path}")
    print(f"Jumlah data: {len(df)}")

    return df

Kode di atas digunakan untuk melakukan proses **ekstraksi fitur tekstur citra dari folder hasil preprocessing secara otomatis** kemudian menyimpan hasil ekstraksi tersebut ke dalam file CSV. Pada bagian ini program membuat fungsi `extract_features_from_preprocessing_folder` yang bertugas membaca seluruh gambar pada folder preprocessing menghitung fitur tekstur menggunakan metode GLCM serta menyusun hasilnya menjadi dataset yang siap digunakan pada tahap analisis atau klasifikasi.

Pada awal fungsi program menentukan lokasi folder preprocessing menggunakan variabel `prepo_path` yang diperoleh dari gabungan folder utama preprocessing dengan nama folder preprocessing tertentu. Folder ini digunakan sebagai sumber data gambar yang akan diproses.

Program kemudian melakukan pengecekan apakah folder preprocessing tersedia atau tidak. Apabila folder tidak ditemukan maka sistem akan menampilkan pesan kesalahan dan proses dihentikan agar program tidak mengalami error akibat lokasi data yang tidak tersedia.

Setelah folder berhasil ditemukan program membuat variabel `rows` yang berfungsi untuk menyimpan seluruh hasil ekstraksi fitur dari setiap gambar. Variabel ini akan digunakan untuk mengumpulkan data sebelum diubah menjadi tabel dataset.

Program selanjutnya melakukan perulangan pada setiap folder kelas yang terdapat di dalam folder preprocessing. Setiap folder kelas dianggap sebagai label data seperti kategori atau jenis objek pada dataset. Pada tahap ini sistem hanya memproses folder dan mengabaikan file lain yang bukan direktori.

Setelah mendapatkan label kelas program membaca seluruh file gambar berdasarkan format yang telah ditentukan sebelumnya seperti JPG JPEG PNG dan BMP. File gambar diurutkan terlebih dahulu agar proses pembacaan data menjadi lebih terstruktur dan konsisten.

Program kemudian menampilkan jumlah gambar yang ditemukan pada setiap folder kelas. Informasi ini digunakan untuk membantu proses pengecekan apakah seluruh data citra telah terbaca dengan benar sebelum ekstraksi fitur dilakukan.

Pada setiap gambar program menggunakan OpenCV untuk membaca citra dalam format grayscale karena metode GLCM bekerja pada citra keabuan. Jika gambar gagal dibaca maka sistem akan menampilkan pesan kesalahan dan melanjutkan proses ke gambar berikutnya tanpa menghentikan keseluruhan program.

Setelah gambar berhasil dibaca program mengubah tipe data citra menjadi `uint8` agar sesuai dengan format yang dibutuhkan pada proses pembentukan matriks GLCM. Tahap ini penting untuk memastikan seluruh gambar dapat diproses dengan baik tanpa error.

Program kemudian menjalankan fungsi `extract_glcm_features` untuk menghitung seluruh fitur tekstur dari gambar berdasarkan beberapa sudut analisis GLCM. Hasil ekstraksi fitur seperti contrast homogeneity dissimilarity entropy ASM energy dan correlation disimpan dalam bentuk data fitur.

Selanjutnya sistem membuat struktur data `row` yang berisi informasi nama file label kelas dan jenis preprocessing yang digunakan. Setelah itu seluruh hasil fitur tekstur ditambahkan ke dalam data tersebut sehingga setiap gambar memiliki satu baris data lengkap yang berisi identitas gambar dan nilai fitur teksturnya.

Seluruh data hasil ekstraksi kemudian dikumpulkan ke dalam variabel `rows` dan diubah menjadi bentuk tabel menggunakan `pandas DataFrame`. Setelah dataset terbentuk program menyimpan hasilnya ke folder output dalam format CSV sesuai nama file yang telah ditentukan sebelumnya.

Pada bagian akhir program menampilkan lokasi penyimpanan file CSV serta jumlah total data yang berhasil diekstraksi. Fungsi ini juga mengembalikan hasil dataset agar dapat digunakan kembali pada proses berikutnya apabila diperlukan.


In [6]:
hasil_ekstraksi = {}

for prepo_name, output_csv_name in PREPO_CONFIG.items():
    print("\n" + "=" * 80)
    print(f"Ekstraksi fitur: {prepo_name}")
    print("=" * 80)

    df = extract_features_from_preprocessing_folder(prepo_name, output_csv_name)

    if df is not None:
        hasil_ekstraksi[prepo_name] = df

print("\nSemua ekstraksi fitur selesai.")


Ekstraksi fitur: prepo1_resize+grayscale
prepo1_resize+grayscale | catterpillar: 70 gambar


prepo1_resize+grayscale | snail: 70 gambar
CSV berhasil disimpan: ..\hasil_ekstraksi\hasil_ekstraksi_prepo1.csv
Jumlah data: 140

Ekstraksi fitur: prepo2_resize+grayscale+median
prepo2_resize+grayscale+median | catterpillar: 70 gambar
prepo2_resize+grayscale+median | snail: 70 gambar
CSV berhasil disimpan: ..\hasil_ekstraksi\hasil_ekstraksi_prepo2.csv
Jumlah data: 140

Ekstraksi fitur: prepo3_resize+grayscale+median+equ
prepo3_resize+grayscale+median+equ | catterpillar: 70 gambar
prepo3_resize+grayscale+median+equ | snail: 70 gambar
CSV berhasil disimpan: ..\hasil_ekstraksi\hasil_ekstraksi_prepo3.csv
Jumlah data: 140

Ekstraksi fitur: prepo4_resize+grayscale+median+sobel
prepo4_resize+grayscale+median+sobel | catterpillar: 70 gambar
prepo4_resize+grayscale+median+sobel | snail: 70 gambar
CSV berhasil disimpan: ..\hasil_ekstraksi\hasil_ekstraksi_prepo4.csv
Jumlah data: 140

Ekstraksi fitur: prepo5_resize+grayscale+median+sobel+thresholding
prepo5_resize+grayscale+median+sobel+thresholdi

Kode di atas digunakan untuk menjalankan proses **ekstraksi fitur tekstur citra pada seluruh folder preprocessing secara otomatis**. Pada bagian ini program melakukan pemanggilan fungsi ekstraksi fitur untuk setiap hasil preprocessing yang telah didefinisikan sebelumnya kemudian menyimpan hasil ekstraksi tersebut agar dapat digunakan pada tahap analisis atau klasifikasi.

Pada awal kode program membuat variabel `hasil_ekstraksi` dalam bentuk dictionary yang digunakan untuk menyimpan seluruh dataset hasil ekstraksi fitur dari masing masing preprocessing. Variabel ini berfungsi sebagai tempat penyimpanan sementara agar hasil ekstraksi dapat diakses kembali tanpa perlu membaca ulang file CSV.

Program kemudian melakukan perulangan pada setiap data yang terdapat di dalam `PREPO_CONFIG`. Perulangan ini mengambil nama folder preprocessing dan nama file output CSV yang telah ditentukan sebelumnya. Dengan cara ini sistem dapat menjalankan proses ekstraksi secara otomatis pada seluruh tahapan preprocessing tanpa perlu menjalankan kode satu per satu secara manual.

Pada setiap proses perulangan program menampilkan informasi proses ekstraksi fitur yang sedang dijalankan. Tampilan ini digunakan untuk memberikan penanda kepada pengguna mengenai preprocessing yang sedang diproses sehingga memudahkan proses monitoring saat program berjalan.

Selanjutnya program memanggil fungsi `extract_features_from_preprocessing_folder` dengan parameter nama preprocessing dan nama file output CSV. Fungsi ini bertugas membaca seluruh gambar pada folder preprocessing menghitung fitur tekstur menggunakan metode GLCM kemudian menyimpan hasil ekstraksi ke dalam file CSV.

Setelah proses ekstraksi selesai program melakukan pengecekan apakah hasil ekstraksi berhasil diperoleh atau tidak. Jika hasil ekstraksi tidak bernilai kosong maka dataset akan disimpan ke dalam variabel `hasil_ekstraksi` menggunakan nama preprocessing sebagai kunci data. Hal ini mempermudah akses terhadap hasil ekstraksi dari setiap metode preprocessing secara terpisah.

Pada bagian akhir program menampilkan informasi bahwa seluruh proses ekstraksi fitur telah selesai dijalankan. Informasi ini menunjukkan bahwa semua folder preprocessing berhasil diproses dan hasil ekstraksi fitur telah disimpan untuk digunakan pada tahapan berikutnya.
